In [2]:
import pandas as pd
import duckdb
import hashlib

In [3]:
# Step 1
df = pd.read_csv("../data/user_agg_sample_1250_users.csv")

print("row count:", len(df))
print("columns:", df.columns.tolist())

row count: 1250
columns: ['user_id', 'impression', 'purchase', 'revenue', 'converted']


In [5]:
# Step 2
con = duckdb.connect("assignment1.duckdb")

con.execute("CREATE SCHEMA IF NOT EXISTS raw_data;")
con.execute("DROP TABLE IF EXISTS raw_data.user_agg_example;")

con.execute("""
CREATE TABLE raw_data.user_agg_example (
    user_id INTEGER,
    impression INTEGER,
    purchase INTEGER,
    revenue DOUBLE,
    converted BOOLEAN
);
""")

# TODO:
# DuckDB에서 CSV 파일을 직접 읽어 테이블에 적재하세요.
# read_csv_auto()를 사용하면 CSV를 SQL에서 바로 읽을 수 있습니다.
# INSERT INTO ... SELECT 형태로 작성해야 합니다.
con.execute("""
INSERT INTO raw_data.user_agg_example
SELECT * FROM read_csv_auto('../data/user_agg_sample_1250_users.csv')
""")

In [6]:
# Step 3
# TODO:
# user_id를 기반으로 SQL에서 variant를 생성하세요.
# HASH(user_id)를 사용해 해시값을 만든 뒤,
# MOD(..., 2)를 적용하여 0 또는 1로 나누도록 작성하세요.
sql_variant_df = con.execute("""
SELECT
    user_id,
    MOD(HASH(user_id), 2) AS variant
FROM raw_data.user_agg_example
LIMIT 10
""").df()

print(sql_variant_df)


   user_id  variant
0    12346        0
1    12348        0
2    12352        0
3    12359        0
4    12360        1
5    12362        0
6    12372        0
7    12378        1
8    12379        0
9    12383        0


In [7]:
# Step 4
# TODO:
# variant별 사용자 수를 집계하세요.
# 내부에서 variant를 먼저 만들고,
# COUNT(DISTINCT user_id)를 사용해 그룹별 인원 수를 계산하세요.
sql_group_df = con.execute("""
SELECT
    variant,
    COUNT(DISTINCT user_id) AS users
FROM (
    SELECT
        user_id,
        MOD(HASH(user_id), 2) AS variant
    FROM raw_data.user_agg_example
)
GROUP BY variant
ORDER BY variant
""").df()

print(sql_group_df)

   variant  users
0        0    628
1        1    622


In [8]:
# Step 5
def split_user(user_id):
    h = hashlib.md5(str(user_id).encode())

    # TODO:
    # md5 해시값을 정수로 변환한 뒤,
    # 2로 나눈 나머지를 반환하세요.
    # SQL의 MOD(HASH(user_id), 2)와 동일한 역할입니다.
    return int(h.hexdigest(),16) % 2

df["variant_50pct"] = df["user_id"].apply(split_user)

print(df["variant_50pct"].value_counts().sort_index())

variant_50pct
0    636
1    614
Name: count, dtype: int64


In [9]:
# Step 6
def split_user_percent(user_id, b_percent=10):
    h = hashlib.md5(str(user_id).encode())

    # TODO:
    # 해시값을 정수로 변환한 뒤 % 100을 적용하여
    # 0~99 범위 값을 만든 다음,
    # b_percent보다 작으면 1, 아니면 0을 반환하세요.
    val = int(h.hexdigest(),16) % 100

    return 1 if val < b_percent else 0

df["variant_10pct"] = df["user_id"].apply(lambda x: split_user_percent(x, 10))

print(df["variant_10pct"].value_counts().sort_index())

variant_10pct
0    1126
1     124
Name: count, dtype: int64


In [10]:
# Step 7 
# TODO:
# variant_50pct 기준으로 그룹을 나누고,
# user_id는 count,
# 나머지 컬럼은 mean으로 집계하세요.
result_50 = df.groupby("variant_50pct").agg({
    "user_id": "count",
    "impression": "mean",
    "purchase": "mean",
    "revenue": "mean",
    "converted": "mean"
})

print(result_50)

               user_id  impression  purchase      revenue  converted
variant_50pct                                                       
0                  636    3.977987  1.040881  1308.070613   0.795597
1                  614    3.887622  1.037459  1599.083160   0.804560
